# FACULTY SOLUTION — Week 13: Factor markets — three labor structures
Computes every answer from the canonical CSVs and asserts it against the golden fixtures.

In [ ]:
import pandas as pd, numpy as np, math, json, itertools
d = {s: pd.read_csv(f'../data/{s}.csv') for s in ['norway_union', 'permian_labor', 'turnaround', 'wage_benchmarks', 'worked_example_labor']}

In [ ]:

def compute(d):
    n = {x.parameter: x.value for x in d['norway_union'].itertuples()}; p = {x.parameter: x.value for x in d['permian_labor'].itertuples()}
    t = {x.parameter: x.value for x in d['turnaround'].itertuples()}
    r = {}
    r['norway_gross_cost_musd'] = n['wage_bill_musd'] * n['union_demand_pct']
    r['norway_after_tax_cost_musd'] = r['norway_gross_cost_musd'] * (1 - n['tax_rate'])
    r['norway_after_tax_share'] = r['norway_after_tax_cost_musd'] / r['norway_gross_cost_musd']
    r['permian_mrp_k'] = p['bbl_per_marginal_worker'] * p['margin_per_bbl'] / 1000
    r['mrp_to_wage'] = r['permian_mrp_k'] / p['market_wage_k']
    r['turnaround_peak_cost_musd'] = t['labor_cost_base_musd'] * t['peak_multiplier']
    r['delay_expected_cost_musd'] = t['labor_cost_base_musd'] + t['outage_probability_if_delayed'] * t['outage_cost_musd']
    r['delay_saving_musd'] = r['turnaround_peak_cost_musd'] - r['delay_expected_cost_musd']
    r['delay_saving_pct'] = r['delay_saving_musd'] / r['turnaround_peak_cost_musd']
    r['asset_health_penalty_pts'] = t['asset_health_penalty_pts']
    return r


In [ ]:
fx = json.load(open('../fixtures/week13_golden.json'))
r = compute(d)
bad = []
for k, v in fx['results'].items():
    a = r[k]
    ok = (str(a) == str(v)) if isinstance(v, str) else math.isclose(float(a), float(v), rel_tol=1e-3, abs_tol=1e-5)
    print(('OK  ' if ok else 'DIFF'), f'{k:36s}', a)
    if not ok: bad.append(k)
assert not bad, bad
print('FACULTY NOTEBOOK REPRODUCES GOLDEN FIXTURES: PASS')